# Calibration of the muon momenta

**Owner: B.** Measure the momentum scale of the muons in the two $\eta$ regions with the
J/ψ peak, and deliver it in `calibration.json` through a pull request. The code goes in
`calibration.py`; this notebook only calls it.

Lines marked `# TODO` are commented out: they need functions that are not written yet.
Uncomment them when the function is in `main` (A, C) or in your branch (B).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import dimuon
import mc
from calibration import ETA_SPLIT, M_JPSI, Calibration, fit_peak

GROUP = 1           # TODO: your group number (1 to 8)
assert 1 <= GROUP <= 8, 'Set GROUP to the group number your teacher gave you (1 to 8)'

## 1. The control sample

Muon pairs with a mass near the J/ψ. How many events? Which columns have missing values?

In [ ]:
control_raw = pd.read_hdf('data/bsmumu.h5', f'group{GROUP}/control')
print(len(control_raw), 'events')
control_raw.head()

In [ ]:
control_raw.isna().sum()

## 2. The MC

Same format as the data, perfectly calibrated, and with the truth: how many J/ψ were
generated.

In [ ]:
mc_control, truth = mc.generate('control', n_bkg=8000, seed=1)
print(len(mc_control), 'events; truth:', truth)
mc_control.head()

Do data and MC agree in the variables that the calibration does not change? How many
events have both muons in the same $\eta$ region?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (col, bins) in zip(axes, [('mu1_eta', np.linspace(2, 5, 31)),
                                  ('mu1_pid', np.linspace(0, 1, 21)),
                                  ('vtx_chi2', np.linspace(0, 20, 21))]):
    ax.hist(control_raw[col], bins, density=True, histtype='step', label='data')
    ax.hist(mc_control[col], bins, density=True, histtype='step', label='MC')
    ax.set_xlabel(col)
axes[0].axvline(ETA_SPLIT, color='grey', lw=0.5)
axes[0].legend();

In [ ]:
for name, df in [('data', control_raw), ('MC', mc_control)]:
    low1, low2 = df['mu1_eta'] < ETA_SPLIT, df['mu2_eta'] < ETA_SPLIT
    print(f'{name:4s}  both low: {(low1 & low2).mean():.2f}  both high: {(~low1 & ~low2).mean():.2f}'
          f'  mixed: {(low1 != low2).mean():.2f}')

## 3. First on a toy peak

Until A and C are merged there is no mass to fit. Write `fit_peak` on a J/ψ peak made
with numpy, where you know the truth: a Gaussian over a flat background.

In [ ]:
rng = np.random.default_rng(0)
toy_truth = {'mean': 3.05, 'width': 0.013}
m_toy = np.concatenate([rng.normal(toy_truth['mean'], toy_truth['width'], 2000),
                        rng.uniform(2.85, 3.25, 1000)])

plt.hist(m_toy, bins=np.linspace(2.85, 3.25, 81), histtype='step')
plt.axvline(M_JPSI, color='grey', lw=0.5)
plt.xlabel(r'$m_{\mu\mu}$ [GeV]');

In [ ]:
# TODO (B): uncomment when fit_peak is written
# mean, error, width = fit_peak(m_toy)
# print(f'mean = {mean:.4f} ± {error:.4f} GeV, width = {1000 * width:.1f} MeV')
# print('truth:', toy_truth)

## 4. Selection and mass

Needs `dimuon.select` (C) and `dimuon.invariant_mass` (A): pull `main` once they are
merged, and uncomment. Where is the peak in the data, and where in the MC? The J/ψ mass
is 3.0969 GeV.

In [ ]:
# TODO (A, C): uncomment when select and invariant_mass are in main
# control = dimuon.select(control_raw)
# m = dimuon.invariant_mass(control)
# m_mc = dimuon.invariant_mass(dimuon.select(mc_control))
# print(f'{len(control)} events selected')

# bins = np.linspace(2.85, 3.25, 81)
# plt.hist(m, bins, density=True, histtype='step', label='data')
# plt.hist(m_mc, bins, density=True, histtype='step', label='MC')
# plt.axvline(M_JPSI, color='grey', lw=0.5)
# plt.xlabel(r'$m_{\mu\mu}$ [GeV]')
# plt.legend();

## 5. First on the MC

The MC is perfectly calibrated: the fit must give scales close to 1. If not, the method is
wrong, not the detector.

In [ ]:
# TODO (B): uncomment when Calibration.fit is written
# Calibration().fit(dimuon.select(mc_control))

## 6. The scales of the data

Save them in `calibration.json`, and open a pull request with it: the analysis team reads
this file.

In [ ]:
# TODO (B)
# cal = Calibration().fit(control)
# cal.save()
# cal

## 7. Check with the mixed events

Events with one muon in each region were **not** used in the fit. Before the correction,
where is their J/ψ peak? After `cal.apply`, is it at 3.0969 GeV? Plot both histograms.

In [ ]:
# TODO (B)